In [ ]:
#### PREPARE DATA FOR WORD2VEC 2026
import pickle
import spacy
from tqdm import tqdm
nlp = spacy.load("syc_ud_syc")

#### simtho_data.pkl is a dictionary consisting of DOC IDs as keys and lists of tokens as values.
all_data = pickle.load(open("simtho_data.pkl", "rb"))

# split result into sentences of 128 tokens each
fake_sentences = []
for k, v in all_data.items():
    # print(v[0:10])
    n = 128
    split = [v[i:i + n] for i in range(0, len(v), n)]
    for s in split:
        fake_sentences.append(s)

# lemmatize each fake sentence
fake_lemmatized_sentences = []
for sentence in tqdm(fake_sentences):
    doc = nlp(" ".join(sentence))
    lemmatized_sentence = [token.lemma_ for token in doc]
    fake_lemmatized_sentences.append(lemmatized_sentence)

pickle.dump(fake_sentences, open("fake_sentences_2026.pkl", "wb"))
pickle.dump(fake_lemmatized_sentences, open("fake_lemmatized_sentences_2026.pkl", "wb"))

In [ ]:
### TRAIN CBOW AND SKIPGRAM WORD2VEC MODELS
from gensim_train import train_gensim

fake_sentences = pickle.load(open("/Users/bulbul/data/SyrBert/fake_sentences_2026.pkl", "rb"))
train_gensim(fake_sentences, 600, 10, 50, 5, "fake-sentences")

In [ ]:
### TRAIN CBOW AND SKIPGRAM WORD2VEC MODELS ON LEMMATIZED DATA
import pickle
from gensim_train import train_gensim

fake_lemmatized_sentences = pickle.load(open("/Users/bulbul/data/SyrBert/fake_lemmatized_sentences_2026.pkl", "rb"))
train_gensim(fake_lemmatized_sentences, 600, 10, 50, 5, "fake-lemmatized-sentences")

In [ ]:


import os
from constants import MODEL_PATH_OSX, OUTPUT_DATA_OSX
from gensim_use import use_model

words = ["ܕܚܠܬܐ", "ܕܐܝܢ", "ܗܝܡܢܘܬܐ", "ܬܘܕܝܬܐ"]
        #  , \
        #  "ܢܘܪܐ", \
        # "ܡܪܢ", "ܐܠܗܐ", \
        # "ܢܫܐ", "ܐܢܬܬܐ", \
        # "ܕܝܘܐ","ܫܐܕܐ",
        # "ܡܫܝܚܐ", "ܝܫܘܥ"]

lemmata = ["ܕܚܠܬܐ", "ܕܐܝܢ", "ܗܝܡܢܘܬܐ", "ܬܘܕܝܬܐ"]
# , \
#          "ܢܘܪܐ", \
#         "ܡܪܝܐ", "ܐܠܗܐ", \
#         "ܢܫܐ", "ܐܢܬܬܐ", \
#         "ܕܝܘܐ","ܫܐܕܐ",
#         "ܡܫܝܚܐ", "ܝܫܘܥ"]

words_vectors = []
lemmata_vectors = []

for word in words:
    use_model(os.path.join(MODEL_PATH_OSX, 'words'), os.path.join(OUTPUT_DATA_OSX, "words"), word, 10)
    use_model(os.path.join(MODEL_PATH_OSX, 'words'), os.path.join(OUTPUT_DATA_OSX, "words"), word, 20)
    words_vectors.append(use_model(os.path.join(MODEL_PATH_OSX, 'words'), os.path.join(OUTPUT_DATA_OSX, "words"), word, 10)[1])

for lemma in lemmata:
    use_model(os.path.join(MODEL_PATH_OSX, 'lemmata'), os.path.join(OUTPUT_DATA_OSX, "lemmata"), lemma, 10)
    use_model(os.path.join(MODEL_PATH_OSX, 'lemmata'), os.path.join(OUTPUT_DATA_OSX, "lemmata"), lemma, 20)
    lemmata_vectors.append(use_model(os.path.join(MODEL_PATH_OSX, 'lemmata'), os.path.join(OUTPUT_DATA_OSX, "lemmata"), lemma, 10)[1])

In [ ]:
### CALCULATE COSINE SIMILARITIES FOR SELECTED WORDS AND LEMMATA
import numpy as np
import pandas as pd
import re
from itertools import combinations
from sklearn.metrics.pairwise import cosine_similarity

final_data = {}

for vector_source in ['words', 'lemmata']:
    cbow_data = {}
    skipgram_data = {}

    cbow_line = []
    skipgram_line = []

    for vector in eval(f"{vector_source}_vectors"):
        for k, v in vector.items():
            if re.match(r".*cbow.*", k):
                cbow_data[v[0]] = v[1]
            else:
                skipgram_data[v[0]] = v[1]

    for k, v in combinations(cbow_data.items(), 2):
        cosine_similarity_syr = cosine_similarity([k[1]], [v[1]])[0][0]
        cbow_line.append(f"{k[0]}/{v[0]}, {cosine_similarity_syr:.5f}")
        print(f"CBOW {vector_source}: Cosine similarity between {k[0]} and {v[0]} \t {cosine_similarity_syr:.5f}")
    final_data[vector_source + "_CBOW"] = cbow_line
    print("--------------------------------")

    for k, v in combinations(skipgram_data.items(), 2):
        cosine_similarity_syr = cosine_similarity([k[1]], [v[1]])[0][0]
        skipgram_line.append(f"{k[0]}/{v[0]}, {cosine_similarity_syr:.5f}")
        print(f"SKIPGRAM {vector_source}: Cosine similarity between {k[0]} and {v[0]} \t {cosine_similarity_syr:.5f}")
    final_data[vector_source + "_skipgram"] = skipgram_line
    print("=================================")

for k, v in final_data.items():
    print(k, v)

In [ ]:
#### EXPORT COSINE SIMILARITIES TO TSV

import pandas as pd
df = pd.DataFrame(final_data)

new_cols = []
new_df = pd.DataFrame()

for col in df.columns:
    new_df[[f"{col}_word", f"{col}_similarity"]] = df[col].str.split(", ", expand=True)

new_df.head()
new_df.to_csv("cosine_similarities.tsv", index=False, header=True, sep="\t")


In [ ]:
#### GENERATE TEXT TYPE STATISTICS

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import re
import seaborn as sb
from glob import glob
from pathlib import Path
from natsort import os_sorted

files = os_sorted(glob("*.tsv"))

all_data = {}

for file in files:
    file_data = {}
    df = pd.read_csv(file, sep="\t")
    df = df.head(15)
    for row in df.iterrows():
        file_data[row[1][0]] = row[1][1]
    all_data[file] = dict(sorted(file_data.items()))

list_dicts = [v for k, v in all_data.items()]

df = pd.DataFrame(list_dicts, index=[re.sub(r"\.tsv$", "", k) for k, v in all_data.items()])
df = df.fillna(0)
df[df.columns] = df[df.columns].div(df[df.columns].sum(axis=1), axis=0).multiply(100)
df.to_csv("target_terms_texttype.csv", sep="\t")

plt.figure(figsize=(8, 8))
ax = sb.heatmap(df, annot=False, fmt="g", cmap="Reds")
ax.invert_yaxis()
plt.show()

